# Allpass FDN completion

For a given feedback matrix **A**, the goal is to construct **b**, **c**, and **d** such that the FDN is uniallpass.

See *Allpass Feedback Delay Networks*, Sebastian J. Schlecht (IEEE Trans. Signal Processing).

— Original MATLAB: Sebastian J. Schlecht, 9 June 2020

## Setup

In [ ]:
import numpy as np
import scipy
import pyFDN

np.random.seed(3)

## Test: complete allpass FDN (full MIMO)

Given **A** from a random orthogonal matrix, use **complete_fdn** with **k = N** to get **B, C, D** so that **V = [A,B;C,D]** is orthogonal. Check with **is_uniallpass**.

In [ ]:
N = 12
num_io = N

U = pyFDN.random_orthogonal(N)
G = np.diag(np.random.rand(N))
A = U @ G

B, C, D, X = pyFDN.complete_fdn(A, k=num_io)
is_a, P = pyFDN.is_uniallpass(A, B, C, D, tol=1e-7)

assert is_a, "Completed system should be uniallpass"

pyFDN.plot_system_matrix(A, B, C, D)

## Test: complete diagonally similar to orthogonal (MIMO)

**A** is diagonally similar to an orthogonal block; use **complete_fdn** with **k=1** and check with **is_uniallpass**.

In [ ]:
N = 3
num_io = 2
X1 = scipy.linalg.block_diag(np.diag(np.random.rand(N)), np.eye(num_io))

V = pyFDN.random_orthogonal(N + num_io)
XVX = np.linalg.solve(X1, V @ X1)
XAX = XVX[:N, :N]

B, C, D, X = pyFDN.complete_fdn(XAX, k=num_io)
is_a, P = pyFDN.is_uniallpass(XAX, B, C, D, tol=1e-7)

assert is_a, "Completed system should be uniallpass"

pyFDN.plot_system_matrix(XAX, B, C, D)

## Test: complete series allpass (SISO)

**A** from Schroeder series allpass; use **complete_fdn** with **k=1** and check with **is_uniallpass**.

In [ ]:
N = 4
g = np.random.uniform(0.5, 0.99, N)
A, _, _, _ = pyFDN.series_allpass(g)


B, C, D, X = pyFDN.complete_fdn(A, k=1)
is_a, P = pyFDN.is_uniallpass(A, B, C, D, tol=1e-7)

assert is_a, "Completed system should be uniallpass"

pyFDN.plot_system_matrix(A, B, C, D)

## Test: nested allpass (SISO)

**A** from nested allpass; use **complete_fdn** with **k=1** and check with **is_uniallpass**. (Completion can sometimes fail for poor singular-value structure.)

In [ ]:
N = 3
g = np.random.uniform(0.5, 0.99, N)
A, _, _, _ = pyFDN.nested_allpass(g)

B, C, D, X = pyFDN.complete_fdn(A, k=1)
is_a, P = pyFDN.is_uniallpass(A, B, C, D, tol=1e-7)

assert is_a, "Completed system should be uniallpass"

pyFDN.plot_system_matrix(A, B, C, D)

## Homogeneous allpass with random admissible X

Random delays and gain matrix **G**, then random admissible diagonal **X** (Python translation of `randAdmissibleHomogeneousAllpass`), and **homogeneous_allpass_fdn(G, X)**. Then complete feedback matrix with **complete_fdn** and check again with **is_uniallpass**.

In [ ]:
# Homogeneous allpass FDN with random admissible diagonal X
N = 4
delays = np.random.randint(1, 31, size=N)  # delays in samples, 1..30

g = 0.99  # global gain per sample
G = np.diag(g ** delays)  # gain matrix

X = pyFDN.rand_admissible_homogeneous_allpass(G, (0.7, 0.999))
R = X @ G @ G

A, b, c, d, U = pyFDN.homogeneous_allpass_fdn(G, X)

is_a0, P0 = pyFDN.is_uniallpass(A, b, c, d, tol=1e-7)
assert is_a0, "Completed system should be uniallpass"

B, C, D, X = pyFDN.complete_fdn(A, k=1)
is_a, P = pyFDN.is_uniallpass(A, B, C, D, tol=1e-7)
assert is_a, "Completed system should be uniallpass"

pyFDN.plot_system_matrix(A, B, C, D)